In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import requests
from google.colab import drive

In [2]:
drive.mount("/content/drive/")

Mounted at /content/drive/


In [3]:
df = pd.read_parquet("/content/drive/MyDrive/Nino/finished_data/complete_df.parquet")
df.drop(columns=["death_label_t+3", "damage_label_t+3"], inplace=True)

In [4]:
# since we're no longer predicting, there is no need for a lead time
df = df.sort_values(by=["GEOID", "date"])
df["damage_label"] = df.groupby("GEOID")["damage_label"].shift(1)
df["death_label"] = df.groupby("GEOID")["death_label"].shift(1)


# create the season column
df["season"] = (df["date"].astype(str).str[5:].astype(int) % 12) // 3
df.columns

Index(['GEOID', 'date', 'damage_label', 'death_label', 'damage_antecedent_60',
       'damage_antecedent_12', 'deaths_antecedent_60', 'deaths_antecedent_12',
       'volumetric_soil_water_layer_3', 'precip_prior3', 'precip_prior6',
       'nino_3.4', 'year', 'population', 'pop_density', 'poverty_rate',
       'wind_speed', 'temperature_2m_anomaly',
       'dewpoint_temperature_2m_anomaly', 'hand_vulnerability', 'dist_coast_m',
       'pct_no_vehicle', 'lat', 'lon', 'season'],
      dtype='object')

In [5]:
county_seasons = df.groupby(["GEOID", "season"]).agg(
    n_months = ("damage_label", "size"),
    n_events = ("damage_label", lambda s: (s > 0).sum()),
    p_damage = ("damage_label", lambda s: (s > 0).mean()),
    total_damage = ("damage_label", "sum"),
    max_damage = ("damage_label", "max"),
    imax_damage = ("damage_label", "idxmax"),

    p_death =  ("death_label", lambda s: (s > 0).mean()),
    total_death = ("death_label", "sum"),
    max_death = ("death_label", "max"),
    imax_death = ("death_label", "idxmax"),
)

county_seasons["max_damage_date"] = df.loc[county_seasons["imax_damage"], "date"].values
county_seasons["max_death_date"] = df.loc[county_seasons["imax_death"], "date"].values
county_seasons.drop(columns=["imax_death", "imax_damage"], inplace=True)

# get quantiles conditional on damage
# reset_index only turns both indices into columns based on their names, while calling unstack first and then reset_index creates a column out of the inner index's values, while converting the outer index to a column by its name
damage_only = (df[df["damage_label"] > 0].groupby(["GEOID", "season"])["damage_label"]
  .quantile([0.5, 0.75, 0.9, 0.95])
  .unstack() # turns the quantiles from a second index into standard columns
  .rename(columns={0.5:"q50", 0.75:"q75", 0.9:"q90", 0.95:"q95"})
  .reset_index())

county_seasons = county_seasons.merge(damage_only, how="left", on=["GEOID", "season"])

# exceedance probabilities, nonconditional so run all over rows
for name, value in [("p_10k", 1e4), ("p_100k", 1e5), ("p_1m", 1e6), ("p_10m", 1e7)]:
  df_copy = df.copy()
  df_copy["damage_binary"] = (df_copy["damage_label"] > value)
  df_copy = df_copy.groupby(["GEOID", "season"])["damage_binary"].mean().rename(name).reset_index()
  county_seasons = county_seasons.merge(df_copy, how="left", on=["GEOID","season"])

county_seasons

,GEOID,season,n_months,n_events,p_damage,total_damage,max_damage,p_death,total_death,max_death,max_damage_date,max_death_date,q50,q75,q90,q95,p_10k,p_100k,p_1m,p_10m
0,01001,0,92,11,0.119565,1.532801e+07,1.497545e+07,0.010870,7.000000,7.000000,2008-02,2023-01,36105.757009,55198.608809,8.730599e+04,7.531378e+06,0.097826,0.010870,0.010870,0.010870
1,01001,1,90,28,0.311111,1.289704e+07,6.416114e+06,0.011111,3.000000,3.000000,2003-04,2011-04,23574.427964,99969.679111,7.011522e+05,2.644210e+06,0.233333,0.077778,0.022222,0.000000
2,01001,2,90,25,0.277778,1.273480e+06,3.768240e+05,0.022222,2.000000,1.000000,1999-06,2005-07,18691.274622,28986.464586,1.304786e+05,2.888091e+05,0.166667,0.044444,0.000000,0.000000
3,01001,3,90,13,0.144444,8.297816e+06,4.444824e+06,0.000000,0.000000,0.000000,2004-09,1996-09,72714.398645,373917.537747,1.568130e+06,2.728931e+06,0.111111,0.055556,0.033333,0.000000
4,01003,0,92,21,0.228261,3.071881e+06,2.035785e+06,0.000000,0.000000,0.000000,2017-01,1996-02,15969.394841,64424.391329,2.146287e+05,3.732959e+05,0.152174,0.043478,0.010870,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12427,56043,3,90,1,0.011111,9.180620e+03,9.180620e+03,0.011111,0.083266,0.083266,2007-09,2017-10,9180.619500,9180.619500,9.180620e+03,9.180620e+03,0.000000,0.000000,0.000000,0.000000
12428,56045,0,92,4,0.043478,3.409800e+05,1.869588e+05,0.000000,0.000000,0.000000,2000-02,1996-02,67433.236703,109060.591078,1.557995e+05,1.713791e+05,0.043478,0.010870,0.000000,0.000000
12429,56045,1,90,9,0.100000,1.129381e+07,1.002938e+07,0.000000,0.000000,0.000000,1997-04,1996-03,19751.104294,304855.356341,2.569850e+06,6.299613e+06,0.077778,0.044444,0.011111,0.011111
12430,56045,2,90,23,0.255556,2.407219e+06,1.188521e+06,0.000000,0.000000,0.000000,2010-07,1996-06,28044.548202,55799.299834,1.628812e+05,3.385148e+05,0.166667,0.055556,0.011111,0.000000


In [6]:
# for each county-season, create features describing what percent of damage is caused by which event cateogry

CONVECTIVE = [
    "Thunderstorm Wind",
    "Hail",
    "Tornado",
    "Lightning",
    "Funnel Cloud",
    "Dust Devil",
]

FLOOD = [
    "Flash Flood",
    "Flood",
    "Heavy Rain",
    "Debris Flow",
]

TROPICAL = [
    "Hurricane (Typhoon)",
    "Tropical Storm",
    "Tropical Depression",
    "Storm Surge/Tide",
]

WINTER = [
    "Winter Storm",
    "Winter Weather",
    "Heavy Snow",
    "Blizzard",
    "Ice Storm",
    "Lake-Effect Snow",
    "Sleet",
    "Extreme Cold/Wind Chill",
    "Cold/Wind Chill",
    "Freezing Fog",
    "Avalanche",
]

WIND = [
    "High Wind",
    "Strong Wind",
    "Dust Storm",
]

COASTAL = [
    "Coastal Flood",
    "Lakeshore Flood",
    "High Surf",
    "Rip Current",
    "Sneakerwave",
    "Seiche",
    "Tsunami",
]

HEAT = [
    "Heat",
    "Excessive Heat",
]

FIRE = [
    "Wildfire",
    "Dense Smoke",
]

OTHER = [
    "Dense Fog",
    "Volcanic Ashfall",
    "Volcanic Ash",
]

EVENT_CATEGORIES = {
    "convective": CONVECTIVE,
    "flood": FLOOD,
    "tropical": TROPICAL,
    "winter": WINTER,
    "wind": WIND,
    "coastal": COASTAL,
    "heat": HEAT,
    "fire": FIRE,
    "other": OTHER,
}

for name, category in EVENT_CATEGORIES.items():
  df_copy = df.copy()
  df_copy = df_copy[df_copy["EVENT_TYPE"].isin(category)]
  df_copy = df_copy.groupby(["GEOID", "season"]).agg(
      total_event_damage = ("damage_label", "sum"),
  ).reset_index()
  df_copy = df_copy.rename(columns={"total_event_damage": f"total_{name}_damage"})

  county_seasons = county_seasons.merge(df_copy, how="left", on=["GEOID", "season"])
  county_seasons[f"{name}_event_share"] = county_seasons[f"total_{name}_damage"] / county_seasons[f"total_damage"]
  county_seasons.drop(columns=[f"total_{name}_damage"], inplace=True)

KeyError: 'EVENT_TYPE'

In [ ]:
county_seasons_json = {}

def nan_to_null(number, digits=0, intCheck=None):
  if pd.isna(number):
    return None
  if intCheck:
    return int(round(float(number)))
  return round(float(number), digits)

for row in county_seasons.to_dict("records"): # makes every df row into a dictionary
  key = f"{row['GEOID']}-{row['season']}"
  county_seasons_json[key] = {
      "n_months" : row["n_months"],
      "n_events" : nan_to_null(row["n_events"], intCheck=True),
      "p_damage": nan_to_null(row["p_damage"], 3) if row["total_damage"] > 0 else None,
      "total_damage": nan_to_null(row["total_damage"], 0, intCheck=True),
      "max_damage": nan_to_null(row["max_damage"], 0, intCheck=True) if row["total_damage"] > 0 else None,
      "max_damage_date": row["max_damage_date"] if row["max_damage"] > 0 else None,
      "quantiles": {
          "q50": nan_to_null(row["q50"], 0, intCheck=True) if row["n_events"] >= 5 else None,
          "q75": nan_to_null(row["q75"], 0, intCheck=True) if row["n_events"] >= 10 else None,
          "q90": nan_to_null(row["q90"], 0, intCheck=True) if row["n_events"] >= 20 else None,
          "q95": nan_to_null(row["q95"], 0, intCheck=True) if row["n_events"] >= 30 else None
      },
      "exceedances": {
          "p_10k": nan_to_null(row["p_10k"], 3),
          "p_100k": nan_to_null(row["p_100k"],3 ),
          "p_1m": nan_to_null(row["p_1m"], 3),
          "p_10m": nan_to_null(row["p_10m"], 3)

      },
      "event_shares": {
          "convective_share": nan_to_null(row["convective_event_share"], 3) if row["total_damage"] > 0 else None,
          "flood_share": nan_to_null(row["flood_event_share"], 3) if row["total_damage"] > 0 else None,
          "tropical_share": nan_to_null(row["tropical_event_share"], 3) if row["total_damage"] > 0 else None,
          "winter_share": nan_to_null(row["winter_event_share"], 3) if row["total_damage"] > 0 else None,
          "wind_share": nan_to_null(row["wind_event_share"], 3) if row["total_damage"] > 0 else None,
          "coastal_share": nan_to_null(row["coastal_event_share"], 3) if row["total_damage"] > 0 else None,
          "heat_share": nan_to_null(row["heat_event_share"], 3) if row["total_damage"] > 0 else None,
          "fire_share": nan_to_null(row["fire_event_share"], 3) if row["total_damage"] > 0 else None,
          "other_share": nan_to_null(row["other_event_share"], 3) if row["total_damage"] > 0 else None,
      },
      "deaths": {
          "p_death": nan_to_null(row["p_death"], 3) if row["total_death"] > 0 else None,
          "total_death": nan_to_null(row["total_death"], 0, intCheck=True),
          "max_death": nan_to_null(row["max_death"], 0, intCheck=True) if row["total_death"] > 0 else None,
          "max_death_date": row["max_death_date"] if row["max_death"] > 0 else None
      }
  }

In [ ]:
county_seasons_json["01001-0"]

In [ ]:
import json
with open("/content/drive/MyDrive/Nino/summary_json/summary_stats.json", "w", encoding="utf-8") as file:
  json.dump(county_seasons_json, file, separators=(",", ":"), ensure_ascii=False)

In [ ]:
# now create the files for each county to get historic distribs
# loop through every geoid, create the dict matrix in the loop and dump it to a folder before moving to next geoid
for county in list(df["GEOID"].unique()):
  county_df = df[df["GEOID"] == county]
  county_month_json = {}
  for row in county_df.to_dict("records"):
    key = row["date"]
    county_month_json[key] = {
      "damage": nan_to_null(row["damage_label"], 0, intCheck=True),
      "death": nan_to_null(row["death_label"], 0, intCheck=True),
      "nino_3.4": nan_to_null(row["nino_3.4"], 2),
      "soil": nan_to_null(row["volumetric_soil_water_layer_3"], 3),
      "precip_3": nan_to_null(row["precip_prior3"], 3),
      "temp": nan_to_null(row["temperature_2m_anomaly"], 3),
    }

  with open(f"/content/drive/MyDrive/Nino/summary_json/counties/{county}_months.json", "w", encoding="utf-8") as file:
    json.dump(county_month_json, file, separators=(",", ":"), ensure_ascii=False)



In [ ]:
df[       'precip_prior3'].sort_values()